# 35. Wrong player audit: how often did the old pipeline track the wrong player?

The old pipeline kept one skeleton per clip (`keypoints_interp`), chosen by box size and motion, with a few manual corrections. It never checked who actually hits the shot. This notebook measures how often the skeleton it kept belonged to someone other than the player performing the shot, and produces the figure of wrong clips for the paper.
It uses the **final selection** (`boxes_sel`, `keypoints_sel_raw`), in which both players are court aware selected, so the old pick can be compared with the two real players.

**The measure**
1. *Automatic.* For every clip, the old skeleton is matched to the near or the far player of the final selection by where the hips are (within 60 px). This says which player the old pipeline really used, including its manual corrections.
2. *Manual.* In a **random sample of clips** (all clips are eligible, seed 0) you decide who performs the shot: the near player (blue), the far player (orange), both, unclear, or neither. The old pick is **hidden** while you judge so it cannot bias you.
3. *Wrong player rate* = share of clearly judged clips where the old skeleton is not the player who performs the shot, with a 95% Wilson interval. Because the old pick was the near player almost always, this is mostly the share of clips where the far player performs the shot.
4. *Illustration.* A second, targeted set of clips (far player moves more than the near player, old pick was the near player) is judged to find clear examples for the figure. **This set is not used for the rate.**

**What you do**
1. Select the venv kernel, run the cells down to the first set of clips (sections 1 and 2 print the automatic result).
2. Section 3: look at the clips and write one letter per clip into `VERDICTS` (instructions in the cell above it).
3. Section 4 prints the wrong player rate. Section 5: judge the second set into `VERDICTS_B`. Section 6 saves the figure.
Keep `N_SAMPLE` the same once you start judging: the clips drawn depend on it.

In [ ]:
import glob
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from clearml import Task

try:
    import cv2
except ImportError:
    cv2 = None

REPO = os.environ.get("BADMINTON_REPO", "/workspace/badminton-honours-project")
DATA = os.environ.get("BADMINTON_DATA", "/workspace/data")
OUTPUTS = os.environ.get("BADMINTON_OUTPUTS", "/workspace/outputs")
RAW_VIDEOS = os.environ.get("BADMINTON_RAW_VIDEOS", f"{DATA}/raw")
EXPORT = f"{OUTPUTS}/results"
os.makedirs(EXPORT, exist_ok=True)
sys.path.insert(0, REPO)
import dataset as ds

SEL_RAW, SEL_BOX = f"{DATA}/keypoints_sel_raw", f"{DATA}/boxes_sel"
OLD_INTERP = f"{DATA}/keypoints_interp"           # old single player skeletons, read only
N_TARGET, CONF_MIN, MATCH_PX = 30, 0.3, 60
N_SAMPLE, SEED = 60, 0                            # clips judged for the rate. 100 gives a tighter interval
N_CAND = 12                                       # clips judged for the illustration figure
NEAR, FAR, OLD, ACT = "#4c72b0", "#dd8452", "#c0392b", "#2e8b57"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

task = Task.init(project_name="BadmintonPoseAnalysis/results", task_name="wrong_player_audit", task_type=Task.TaskTypes.data_processing,
                 tags=["analysis", "two_player", "audit"], reuse_last_task_id=False, auto_connect_frameworks={"matplotlib": False})
logger = task.get_logger()

## 1. The clips

All clips used in the experiments (the same filtered manifest as notebooks 32 and 33) that have a final selection and an old skeleton.

In [ ]:
log_files = sorted(f for f in glob.glob(f"{DATA}/extraction_sel_log*.csv") if os.path.getsize(f) > 0)
sel = pd.concat([pd.read_csv(f) for f in log_files], ignore_index=True).drop_duplicates("filepath", keep="last")
sel = sel[sel["status"] == "ok"].set_index("filepath")

manifest = ds.load_filtered_manifest(f"{DATA}/split_manifest.csv", f"{DATA}/extraction_log.csv",
                                     f"{DATA}/keypoints_interp_log.csv", 5)
pop = manifest[manifest["filepath"].isin(sel.index)].copy()
pop["old_exists"] = [os.path.exists(f"{OLD_INTERP}/{c}/{os.path.splitext(os.path.basename(f))[0]}.npy") for f, c in zip(pop["filepath"], pop["class_name"])]
print(f"manifest clips {len(manifest)}; with a final selection {len(pop)}; without an old skeleton file {int((~pop['old_exists']).sum())}")
pop = pop[pop["old_exists"]].reset_index(drop=True)
pop["span_first"] = sel.loc[pop["filepath"], "span_first"].to_numpy()
pop["span_last"] = sel.loc[pop["filepath"], "span_last"].to_numpy()
pop["n_persons"] = sel.loc[pop["filepath"], "n_persons"].to_numpy()
pop["cls"] = pop["class_name"].str.split("_", n=1).str[1]
popi = pop.set_index("filepath")
print(f"population: {len(pop)} clips, {int((pop['n_persons'] == 2).sum())} with both players found ({(pop['n_persons'] == 2).mean():.1%})")

## 2. Which player did the old pipeline really use? (automatic, all clips)

The old skeleton is compared with the near and the far player of the final selection by the median hip position over the frames where both hips are visible, and assigned to the closer one if it is within 60 px; otherwise `neither`. This covers the old pipeline's manual and wrist speed corrections too.
Wrist speed (fastest wrist, per frame, as a fraction of the player's height) is also computed for each player; it is used only to find candidate clips for the illustration figure.

In [ ]:
def stem_of(fp):
    return os.path.splitext(os.path.basename(fp))[0]


def median_hip(kp):
    hips = kp[:, [11, 12], :]
    ok = (hips[:, :, 2] > CONF_MIN).all(axis=1)
    if not ok.any():
        return np.array([np.nan, np.nan])
    return np.median(hips[ok, :, :2].mean(axis=1), axis=0)


def wrist_speed(kp, box, pres):
    idx = np.where(pres)[0]
    if len(idx) < 2:
        return np.nan
    h = np.median(box[idx, 3] - box[idx, 1])
    best = 0.0
    for w in (9, 10):
        for a, b in zip(idx[:-1], idx[1:]):
            if b == a + 1 and kp[a, w, 2] > CONF_MIN and kp[b, w, 2] > CONF_MIN:
                best = max(best, float(np.linalg.norm(kp[b, w, :2] - kp[a, w, :2]) / max(h, 1.0)))
    return best


rows = []
for fp, cls in zip(pop["filepath"], pop["class_name"]):
    stem = stem_of(fp)
    kps = np.load(f"{SEL_RAW}/{cls}/{stem}.npy")
    boxes = np.load(f"{SEL_BOX}/{cls}/{stem}.npy")
    h_old = median_hip(np.load(f"{OLD_INTERP}/{cls}/{stem}.npy"))
    d, spd = [], []
    for p in (0, 1):
        pres = ~np.isnan(boxes[p, :, 0])
        hp = median_hip(kps[p][pres]) if pres.any() else np.array([np.nan, np.nan])
        d.append(float(np.linalg.norm(h_old - hp)))
        spd.append(wrist_speed(kps[p], boxes[p], pres))
    dn = [np.inf if np.isnan(x) else x for x in d]
    closest = min(dn)
    if np.isnan(h_old).any() or closest == np.inf:
        old_is = "unknown"
    elif closest >= MATCH_PX:
        old_is = "neither"
    else:
        old_is = "near" if dn[0] <= dn[1] else "far"
    rows.append({"filepath": fp, "old_is": old_is, "dist_near": d[0], "dist_far": d[1], "speed_near": spd[0], "speed_far": spd[1]})
auto = pd.DataFrame(rows).set_index("filepath")
auto["split"] = popi.loc[auto.index, "split"]
auto["cls"] = popi.loc[auto.index, "cls"]
auto["far_found"] = popi.loc[auto.index, "n_persons"] == 2
auto.to_csv(f"{EXPORT}/old_pick_vs_selection.csv")

share = auto["old_is"].value_counts()
print("the old skeleton belongs to:")
print(pd.DataFrame({"clips": share, "share": (share / len(auto)).round(4)}).to_string())
matched = auto[auto["old_is"].isin(["near", "far"])]
print(f"\nmatched to a selected player: {len(matched) / len(auto):.1%}")
if len(matched) / len(auto) < 0.9:
    print("WARNING: under 90% matched. Check that keypoints_interp and keypoints_sel_raw use the same pixel coordinates before trusting this.")
by_split = pd.crosstab(auto["split"], auto["old_is"], normalize="index").round(3)
by_class = pd.crosstab(auto["cls"], auto["old_is"], normalize="index").round(3).assign(clips=auto["cls"].value_counts())
display(by_split)
display(by_class)
shares_tbl = pd.DataFrame({"clips": share, "share": (share / len(auto)).round(4)}).reset_index().rename(columns={"index": "old skeleton belongs to", "old_is": "old skeleton belongs to"})
logger.report_table("old skeleton belongs to, all clips", "audit", iteration=0, table_plot=shares_tbl)
for name, val in share.items():
    logger.report_scalar("old skeleton share (all clips)", str(name), float(val / len(auto)), iteration=0)
logger.report_table("old skeleton belongs to, by class", "audit", iteration=0, table_plot=by_class.reset_index())

## 3. Random sample: who performs the shot?

`N_SAMPLE` clips drawn at random from all clips (seed 0, so the same clips every run). Each row of pictures is one clip, five frames from left to right in time. **Blue is the player nearer the camera, orange the one further away.** The old pipeline's choice is not shown.

In [ ]:
def draw_person(ax, kp, box, colour):
    if box is None or np.isnan(box[0]):
        return
    vis = kp[:, 2] > CONF_MIN
    for a, b in ds.COCO_SKELETON:
        if vis[a] and vis[b]:
            ax.plot([kp[a, 0], kp[b, 0]], [kp[a, 1], kp[b, 1]], color=colour, lw=1.6)
    ax.scatter(kp[vis, 0], kp[vis, 1], s=7, color=colour, zorder=3)
    ax.add_patch(Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], fill=False, ec=colour, lw=1.1, ls="--"))


def draw_clip(axes, fp, slots, mark=None):
    """mark = (old_is, actor) draws the old pipeline's choice (red box) and who hits the shot (green label)."""
    r = popi.loc[fp]
    cls, stem = r["class_name"], stem_of(fp)
    kps = np.load(f"{SEL_RAW}/{cls}/{stem}.npy")
    boxes = np.load(f"{SEL_BOX}/{cls}/{stem}.npy")
    targets = np.linspace(r["span_first"], r["span_last"], N_TARGET).round().astype(int)
    path = f"{RAW_VIDEOS}/{fp}"
    cap = cv2.VideoCapture(path) if (cv2 is not None and os.path.exists(path)) else None
    okb = boxes[~np.isnan(boxes[:, :, 0])]
    if len(okb) == 0:
        for ax in axes:
            ax.axis("off")
        return
    x0, y0 = okb[:, :2].min(axis=0)
    x1, y1 = okb[:, 2:].max(axis=0)
    mx, my = 0.08 * (x1 - x0), 0.12 * (y1 - y0)
    for ax, t in zip(axes, slots):
        if cap is not None:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(targets[t]))
            ok, bgr = cap.read()
            if ok:
                ax.imshow(bgr[..., ::-1])
        draw_person(ax, kps[0, t], boxes[0, t], NEAR)
        draw_person(ax, kps[1, t], boxes[1, t], FAR)
        if mark is not None:
            old_is, actor = mark
            for name, p in (("near", 0), ("far", 1)):
                b = boxes[p, t]
                if np.isnan(b[0]):
                    continue
                if old_is == name:
                    ax.add_patch(Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec=OLD, lw=2.4))
                    ax.text(b[0], b[1] - 5, "old pipeline", color=OLD, fontsize=7, weight="bold")
                if actor == name:
                    ax.text(b[0], b[3] + 14, "hits the shot", color=ACT, fontsize=7, weight="bold")
            if old_is in ("neither", "unknown"):
                ax.text(x0, y0 - 5, "old pick is neither player", color=OLD, fontsize=7, weight="bold")
        ax.set_xlim(x0 - mx, x1 + mx)
        ax.set_ylim(y1 + my, y0 - my)
        ax.set_aspect("equal")
        ax.axis("off")
    if cap is not None:
        cap.release()


def show_clip(fp, tag, slots=(0, 7, 14, 21, 29), mark=None):
    fig, axes = plt.subplots(1, len(slots), figsize=(2.9 * len(slots), 3.4))
    draw_clip(axes, fp, slots, mark)
    for ax, t in zip(axes, slots):
        ax.set_title(f"frame {t}", fontsize=7)
    fig.suptitle(f"{tag}   {popi.loc[fp, 'cls']}   (blue = near, orange = far)", fontsize=9, y=1.04)
    plt.show()


sample = pop.sample(N_SAMPLE, random_state=SEED)["filepath"].tolist()


def show_range(a, b):
    for i, fp in enumerate(sample[a:b], start=a):
        show_clip(fp, f"#{i}")


if cv2 is None or not os.path.isdir(RAW_VIDEOS):
    print(f"no cv2 or no video folder at {RAW_VIDEOS}: clips will be drawn on a blank canvas. Set BADMINTON_RAW_VIDEOS to the folder with the source clips.")
print(f"{len(sample)} clips to judge, in three cells of 20")

In [ ]:
show_range(0, 20)

In [ ]:
show_range(20, 40)

In [ ]:
show_range(40, 60)

### Your verdicts

One letter per clip, in the order of the numbers in the titles (#0, #1, ...), separated by spaces. Who performs the labelled shot, the player who swings at the shuttle?

| letter | meaning |
|---|---|
| `n` | the **near** player (blue) |
| `f` | the **far** player (orange) |
| `b` | both players swing inside the clip |
| `u` | unclear, you cannot tell |
| `x` | neither blue nor orange (a bystander, or the hitter is not tracked) |

Use the class name in the title as a clue (a serve is hit by the server). Judge from the frames only, in order, left to right. The cell checks that the number of letters equals `N_SAMPLE`.

In [ ]:
VERDICTS = "n f f n f n n n f n n n n n f f n f f f f f n n f f f f f f u f n n n f n f f f f f f f f u f n f n f u f f f n n n n n"

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return centre - half, centre + half


CODE = {"n": "near", "f": "far", "b": "both", "u": "unclear", "x": "neither"}
audit_result = None
codes = VERDICTS.split()
if not codes:
    print("VERDICTS is empty: fill it in first.")
else:
    assert len(codes) == N_SAMPLE, f"{len(codes)} letters for {N_SAMPLE} clips"
    assert set(codes) <= set(CODE), f"unknown letters: {sorted(set(codes) - set(CODE))}"
    v = pd.DataFrame({"number": range(N_SAMPLE), "filepath": sample, "actor": [CODE[c] for c in codes]})
    v["old_is"] = auto.loc[v["filepath"], "old_is"].to_numpy()
    v["cls"] = popi.loc[v["filepath"], "cls"].to_numpy()
    v.to_csv(f"{EXPORT}/wrong_player_audit_verdicts.csv", index=False)

    print("who performs the shot (your verdicts):")
    print(v["actor"].value_counts().to_string())
    clear = v[v["actor"].isin(["near", "far"]) & (v["old_is"] != "unknown")].copy()
    clear["wrong"] = clear["old_is"] != clear["actor"]            # 'neither' counts as wrong: the old skeleton is not the actor
    k, n = int(clear["wrong"].sum()), len(clear)
    lo, hi = wilson(k, n)
    far_k = int((clear["actor"] == "far").sum())
    flo, fhi = wilson(far_k, n)
    print(f"\nthe far player performs the shot: {far_k} of {n} clear clips = {far_k / n:.1%} (95% interval {flo:.1%} to {fhi:.1%})")
    print(f"OLD PIPELINE TRACKED THE WRONG PLAYER: {k} of {n} clear clips = {k / n:.1%} (95% interval {lo:.1%} to {hi:.1%})")
    allj = len(v)
    extra = int(v["actor"].isin(["both", "unclear", "neither"]).sum())
    print(f"bounds over all {allj} clips if the {extra} unclear, both or neither clips count as right / as wrong: {k / allj:.1%} to {(k + extra) / allj:.1%}")
    print(f"scaled to the {len(pop)} clips: about {round(len(pop) * k / n)} wrong (interval {round(len(pop) * lo)} to {round(len(pop) * hi)})")
    print("\nwho performs the shot against the skeleton the old pipeline used:")
    print(pd.crosstab(v["actor"], v["old_is"]).to_string())
    audit_result = {"n_population": int(len(pop)), "n_sample": int(N_SAMPLE), "seed": SEED, "verdict_counts": v["actor"].value_counts().to_dict(),
                    "n_clear": n, "n_wrong": k, "wrong_rate": k / n, "wrong_ci95": [lo, hi],
                    "far_hits_rate": far_k / n, "far_hits_ci95": [flo, fhi],
                    "bounds_all_judged": [k / allj, (k + extra) / allj],
                    "old_skeleton_belongs_to": auto["old_is"].value_counts().to_dict()}
    with open(f"{EXPORT}/wrong_player_audit.json", "w") as f:
        json.dump(audit_result, f, indent=2)
    logger.report_table("audit verdicts", "audit", iteration=0, table_plot=v)
    summary_tbl = pd.DataFrame([
        {"measure": "clips judged", "value": N_SAMPLE},
        {"measure": "clear clips (near or far)", "value": n},
        {"measure": "old pipeline wrong player", "value": f"{k} ({k / n:.1%}, 95% {lo:.1%} to {hi:.1%})"},
        {"measure": "far player performs the shot", "value": f"{far_k} ({far_k / n:.1%}, 95% {flo:.1%} to {fhi:.1%})"},
        {"measure": "bounds over all judged clips", "value": f"{k / allj:.1%} to {(k + extra) / allj:.1%}"},
        {"measure": "scaled to all clips", "value": f"about {round(len(pop) * k / n)} of {len(pop)}"},
    ])
    logger.report_table("wrong player audit summary", "audit", iteration=0, table_plot=summary_tbl)
    for name, val in (("wrong rate", k / n), ("wrong rate ci low", lo), ("wrong rate ci high", hi),
                      ("far hits rate", far_k / n), ("far hits ci low", flo), ("far hits ci high", fhi)):
        logger.report_scalar("wrong player audit", name, float(val), iteration=0)
    task.connect({"n_sample": N_SAMPLE, "seed": SEED, "match_px": MATCH_PX, "n_population": int(len(pop))})
    task.upload_artifact("wrong_player_audit", artifact_object=f"{EXPORT}/wrong_player_audit.json")
    task.upload_artifact("audit_verdicts", artifact_object=f"{EXPORT}/wrong_player_audit_verdicts.csv")

## 4. Examples for the figure (not used for the rate)

Random sampling finds few wrong clips if they are rare, so a second set is drawn from clips where the old pick was the near player but the **far player's wrist moves faster than the near player's**, which is where a far player hitting the shot is most likely. These clips are for illustration only; the rate above comes only from the random sample.

In [ ]:
cand = auto[(auto["old_is"] == "near") & auto["far_found"] & (auto["speed_far"] > auto["speed_near"]) & ~auto.index.isin(sample)].copy()
cand["ratio"] = cand["speed_far"] / cand["speed_near"].clip(lower=1e-6)
cand = cand.sort_values("ratio", ascending=False).head(200)
cand_files = cand.sample(min(N_CAND, len(cand)), random_state=1).index.tolist()
print(f"{len(cand)} candidate clips; showing {len(cand_files)}")
for i, fp in enumerate(cand_files):
    show_clip(fp, f"B{i}")

Judge these the same way, one letter per clip (`n f b u x`), in the order B0, B1, ...

In [ ]:
VERDICTS_B = "f n n f f f n n f f n n"

## 5. The figure of wrong clips

Takes clips judged wrong from the random sample and from the illustration set (those where the old pick was the near player and the far player hits the shot, or the other way round), and draws up to four: the old pipeline's choice in red, the player who hits the shot in green. Saved to `results/wrong_player_examples.png` for the paper.

In [ ]:
from matplotlib.patches import Patch

examples = []
if audit_result is not None:
    for _, r in clear[clear["wrong"]].iterrows():
        examples.append((r["filepath"], r["actor"], r["old_is"], "random sample"))
cb = VERDICTS_B.split()
if cb:
    assert len(cb) == len(cand_files), f"{len(cb)} letters for {len(cand_files)} clips"
    assert set(cb) <= set(CODE), f"unknown letters: {sorted(set(cb) - set(CODE))}"
    for fp, c in zip(cand_files, cb):
        actor = CODE[c]
        if actor in ("near", "far") and auto.loc[fp, "old_is"] in ("near", "far") and auto.loc[fp, "old_is"] != actor:
            examples.append((fp, actor, auto.loc[fp, "old_is"], "targeted set"))
print(f"{len(examples)} wrong clips available ({sum(e[3] == 'random sample' for e in examples)} from the random sample)")

N_EXAMPLES = 4                                    # rows in the figure
SLOTS = (2, 10, 18, 26)                           # frames shown, left to right in time
TAG = dict(boxstyle="round,pad=0.25", fc="white", alpha=0.92)


def tag(ax, b, p, text, colour):
    """Label with a white background: above the near player's box, below the far player's box."""
    if p == 0:
        ax.text(b[0], b[1] - 8, text, color=colour, fontsize=9, weight="bold", va="bottom", ha="left", bbox={**TAG, "ec": colour}, zorder=10)
    else:
        ax.text(b[0], b[3] + 8, text, color=colour, fontsize=9, weight="bold", va="top", ha="left", bbox={**TAG, "ec": colour}, zorder=10)


def draw_example(axes, fp, old_is, actor):
    r = popi.loc[fp]
    cls, stem = r["class_name"], stem_of(fp)
    kps = np.load(f"{SEL_RAW}/{cls}/{stem}.npy")
    boxes = np.load(f"{SEL_BOX}/{cls}/{stem}.npy")
    targets = np.linspace(r["span_first"], r["span_last"], N_TARGET).round().astype(int)
    path = f"{RAW_VIDEOS}/{fp}"
    cap = cv2.VideoCapture(path) if (cv2 is not None and os.path.exists(path)) else None
    okb = boxes[~np.isnan(boxes[:, :, 0])]
    x0, y0 = okb[:, :2].min(axis=0)
    x1, y1 = okb[:, 2:].max(axis=0)
    mx, my = 0.06 * (x1 - x0), 0.14 * (y1 - y0)
    for k, (ax, t) in enumerate(zip(axes, SLOTS)):
        if cap is not None:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(targets[t]))
            ok, bgr = cap.read()
            if ok:
                ax.imshow(bgr[..., ::-1])
        draw_person(ax, kps[0, t], boxes[0, t], NEAR)
        draw_person(ax, kps[1, t], boxes[1, t], FAR)
        for name, p in (("near", 0), ("far", 1)):
            b = boxes[p, t]
            if np.isnan(b[0]):
                continue
            if actor == name:                                            # who really hits the shot: green
                ax.add_patch(Rectangle((b[0] - 3, b[1] - 3), b[2] - b[0] + 6, b[3] - b[1] + 6, fill=False, ec=ACT, lw=2.6, zorder=8))
                if k == 0:
                    tag(ax, b, p, "hits the shot", ACT)
            if old_is == name:                                           # what the old pipeline tracked: red
                ax.add_patch(Rectangle((b[0] - 3, b[1] - 3), b[2] - b[0] + 6, b[3] - b[1] + 6, fill=False, ec=OLD, lw=2.6, zorder=8))
                if k == 0:
                    tag(ax, b, p, "old pipeline", OLD)
        if old_is == "neither" and k == 0:
            ax.text(x0, y0 - 8, "old pick matches neither player", color=OLD, fontsize=9, weight="bold", va="bottom", bbox={**TAG, "ec": OLD}, zorder=10)
        ax.set_xlim(x0 - mx, x1 + mx)
        ax.set_ylim(y1 + my, y0 - my)
        ax.set_aspect("equal")
        ax.axis("off")
    if cap is not None:
        cap.release()


if examples:
    chosen, seen_cls = [], set()
    for e in examples:                      # one clip per class first, so the figure shows different shots
        c = popi.loc[e[0], "cls"]
        if c not in seen_cls:
            chosen.append(e)
            seen_cls.add(c)
    chosen = chosen[:N_EXAMPLES]
    fig, axs = plt.subplots(len(chosen), len(SLOTS), figsize=(3.2 * len(SLOTS), 3.6 * len(chosen)), squeeze=False)
    for row, (fp, actor, old_is, src_set) in zip(axs, chosen):
        draw_example(row, fp, old_is, actor)
        tracked = "someone else" if old_is == "neither" else f"the {old_is} player"
        row[0].set_title(f"{popi.loc[fp, 'cls']}: the old pipeline tracked {tracked}, but the {actor} player hits the shot",
                         fontsize=11, loc="left", weight="bold", pad=10)
    fig.legend(handles=[Patch(fc="none", ec=OLD, lw=2.6, label="player tracked by the old pipeline"),
                        Patch(fc="none", ec=ACT, lw=2.6, label="player who hits the shot"),
                        Patch(fc=NEAR, label="near player skeleton"), Patch(fc=FAR, label="far player skeleton")],
               loc="lower center", ncol=4, frameon=False, fontsize=10)
    fig.tight_layout(rect=(0, 0.035, 1, 1), h_pad=2.0)
    fig.savefig(f"{EXPORT}/wrong_player_examples.png", dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    logger.report_matplotlib_figure("wrong player examples", "audit", figure=fig, iteration=0, report_interactive=False)
    task.upload_artifact("wrong_player_examples", artifact_object=f"{EXPORT}/wrong_player_examples.png")
    print("saved", f"{EXPORT}/wrong_player_examples.png")
else:
    print("no wrong clips yet: fill VERDICTS and VERDICTS_B")

In [ ]:
both_found = auto[auto["far_found"] & auto["speed_near"].notna() & auto["speed_far"].notna()].copy()
both_found["far_faster"] = both_found["speed_far"] > both_found["speed_near"]
print(f"far player's wrist is faster in {both_found['far_faster'].mean():.1%} of the {len(both_found)} clips where both players were found")
print("\nshare of clips where the far wrist is faster, by class:")
print((100 * both_found.groupby("cls")["far_faster"].mean()).round(1).sort_values().to_string())

s = v[v["actor"].isin(["near", "far"])].join(auto[["speed_near", "speed_far"]], on="filepath")
s["wrist rule says"] = np.where(s["speed_far"] > s["speed_near"], "far", "near")
print("\nyour verdicts against the wrist speed rule (a crude proxy, not ground truth):")
print(pd.crosstab(s["actor"], s["wrist rule says"]).to_string())
print(f"agreement: {(s['actor'] == s['wrist rule says']).mean():.1%} over {len(s)} clips")

## What to report

- **Automatic (section 2):** the share of clips where the old skeleton belonged to the near player, the far player, or neither. This is exact for all clips.
- **Wrong player rate (section 3):** the share of clearly judged random clips where the old skeleton was not the player performing the shot, with its 95% interval and the bounds that treat unclear clips both ways. State the sample size and the seed.
- **Figure:** `results/wrong_player_examples.png`. Say in the caption that the examples come from the targeted set where applicable.
- Judged by one person; if your supervisor judges a subset of the same clips, the agreement between you is worth reporting.

In [ ]:
task.upload_artifact('old_pick_vs_selection', artifact_object=f'{EXPORT}/old_pick_vs_selection.csv')
task.flush(wait_for_uploads=True)
print('logged to ClearML: BadmintonPoseAnalysis/results / wrong_player_audit')